##### Copyright 2026 Google LLC.

In [1]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# MedGemma - Inference using Ollama

Author: Sitam Meur

*   GitHub: [github.com/sitammeur](https://github.com/sitammeur/)
*   X: [@sitammeur](https://x.com/sitammeur)

Description: This notebook demonstrates how you can run inference on [MedGemma](https://developers.google.com/health-ai-developer-foundations/medgemma/model-card) using  [Ollama](https://github.com/ollama/ollama). The [Ollama Python](https://github.com/ollama/ollama-python) library provides the easiest way to integrate Python 3.8+ projects with Ollama.

<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/experiments/MedGemma/[MedGemma]Inference_using_Ollama.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
</table>

## Setup

### Select the Colab runtime
To complete this tutorial, you'll need to have a Colab runtime with sufficient resources to run the [MedGemma 4b](https://huggingface.co/google/medgemma-1.5-4b-it) model. In this case, you can use a T4 GPU:

1. In the upper-right of the Colab window, select **▾ (Additional connection options)**.
2. Select **Change runtime type**.
3. Under **Hardware accelerator**, select **T4 GPU**.

## Installation

Install Ollama through the offical installation script.

In [2]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Install Ollama Python library through the official Python client for Ollama.

In [3]:
!pip install -q ollama

## Start Ollama

Start Ollama in background using nohup.

In [4]:
!nohup ollama serve > ollama.log &

## Prerequisites

*   Ollama should be installed and running. (This was already completed in previous steps.)
*   Pull the [medgemma](https://ollama.com/library/medgemma1.5) model to use with the library: `ollama pull medgemma1.5:4b`
    *  See [Ollama.com](https://ollama.com/) for more information on the models available.

In [5]:
import time
time.sleep(3)  # Wait for Ollama server to be ready

import ollama

try:
    ollama.pull('medgemma1.5:4b')
except ollama.ResponseError as e:
    print('Error pulling model:', e.error)

## Get Image

You can either directly define the image path i.e., `image_url = /path/image.png` or convert into the bytesIO format.

In [6]:
import requests
from IPython.display import display, Image

image_url = "https://www.clinicaladvisor.com/wp-content/uploads/sites/11/2018/12/lungs_0913newsline_451310.jpg"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
}

image_bytes = requests.get(image_url, headers=headers).content

# Visualize
display(Image(data=image_bytes))

## Define the Prompt Template

In [7]:
query = """Based on the attached chest X-ray,
can you analyze and tell me whether the patient is more likely a smoker or a non-smoker?"""

In [8]:
system = """
You are a medical imaging expert AI trained to analyze chest X-rays and identify potential indicators of smoking history.
Given a chest X-ray image, you must determine whether the scan is more likely to belong to a smoker or non-smoker based on clinical radiological signs such as:

- Lung hyperinflation
- Bullae or blebs
- Increased lung lucency
- Irregular reticular or nodular patterns
- Upper lobe cystic changes
- Emphysematous changes or interstitial thickening

Always include a step-by-step explanation based on observed patterns in the image, and mention the level of confidence (e.g., high, moderate, low).
Be cautious not to make a definitive diagnosis but rather offer a likelihood-based reasoning.
"""

## Execute the Prompt

In [9]:
output = ollama.chat(
    model="medgemma1.5:4b",
    messages=[
        {
            "role": "system",
            "content": system
        },
        {
            "role": "user",
            "content": query,
            "images": [image_bytes]
        }
    ]
)

response = output.message.content
print(response)

### Async Usage

To make asynchronous requests, use the `AsyncClient` class; to enable response streaming, set `stream=True`.

In [10]:
import asyncio
import nest_asyncio
from ollama import AsyncClient

nest_asyncio.apply()


async def generate():
    """Asynchronously generates a response to a given prompt using the AsyncClient."""
    # Create an instance of the AsyncClient
    client = AsyncClient()

    # Send a request to generate a response to the prompt
    messages=[
        {
            "role": "system",
            "content": system
        },
        {
            "role": "user",
            "content": query,
            "images": [image_bytes]
        }
    ]

    async for part in await AsyncClient().chat(
        model="medgemma1.5:4b", messages=messages, stream=True
    ):
        print(part["message"]["content"], end="", flush=True)

# Run the generate function
asyncio.run(generate())

## Conclusion

Congratulations! You have successfully run inference on MedGemma. You can now integrate this into your Python projects.